In [5]:
from stormpy import export_to_drn
%load_ext autoreload
%autoreload 2
import sys
import os
print(os.getpid())
sys.path.append('..')

from verimon.logger import setup_logging

setup_logging()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
674305


In [6]:
from verimon import loaders

experiment_premise_a3 = ("../tests/premise/airportA-3.nm",  "DMAX=3,PMAX=3", 'P>0.5 [F<2 "crash" ]', "crash", 8)
experiment_premise_refuel = ("../tests/premise/refuel.nm", "N=4,ENERGY=10", "P>0.5 [F<=5 \"empty\"]", "empty", 14)
experiment_premise_evade = ("../tests/premise/evade-monitoring.nm", "N=5,RADIUS=3", "P>0.5 [F<=5 \"crash\"]", "crash", 10)

test_file, constants, spec, good_label, horizon = experiment_premise_refuel

initial_observation, observations, mc, expr_manager = loaders.pomdp_to_mc(test_file, constants)

DEBUG:2024-11-26 16:56:56,620 - (0.00s) - loaders.py - Finished loading original pomdp with 219 observations 
DEBUG:2024-11-26 16:56:56,628 - (0.01s) - loaders.py - Finished assigning labels to states 
DEBUG:2024-11-26 16:56:56,630 - (0.00s) - loaders.py - Finished creating new transitions 
DEBUG:2024-11-26 16:56:56,667 - (0.04s) - loaders.py - transformed POMDP to stormpy DTMC 


In [7]:
from stormvogel.show import show
from stormvogel.mapping import stormpy_to_stormvogel
import stormvogel

stormvogel.communication_server.enable_server = False

print(mc)
mc_sv = stormpy_to_stormvogel(mc)
# show(mc_sv)

-------------------------------------------------------------- 
Model type: 	DTMC (sparse)
States: 	699
Transitions: 	19745
Reward Models:  none
State Labels: 	225 labels
   * {ax: 3,ay: 3,fuelempty: 1,fuelfull: 0,meter: 5,start: true} -> 5 item(s)
   * {ax: 0,ay: 1,fuelempty: 1,fuelfull: 0,meter: 1,start: true} -> 3 item(s)
   * {ax: 1,ay: 1,fuelempty: 1,fuelfull: 0,meter: 5,start: true} -> 5 item(s)
   * {ax: 0,ay: 2,fuelempty: 1,fuelfull: 0,meter: 9,start: true} -> 3 item(s)
   * {ax: 3,ay: 3,fuelempty: 1,fuelfull: 0,meter: 2,start: true} -> 4 item(s)
   * {ax: 0,ay: 2,fuelempty: 1,fuelfull: 0,meter: 4,start: true} -> 5 item(s)
   * {ax: 2,ay: 3,fuelempty: 1,fuelfull: 0,meter: 8,start: true} -> 3 item(s)
   * {ax: 3,ay: 1,fuelempty: 1,fuelfull: 0,meter: 5,start: true} -> 5 item(s)
   * {ax: 3,ay: 2,fuelempty: 0,fuelfull: 0,meter: 1,start: true} -> 1 item(s)
   * {ax: 2,ay: 3,fuelempty: 0,fuelfull: 0,meter: 2,start: true} -> 1 item(s)
   * {ax: 3,ay: 1,fuelempty: 1,fuelfull: 0,meter:

In [ ]:
from aalpy import run_Lstar, Dfa
from verimon.MonitorLearning import FilteringSUL, VerimonEqOracle

setup_logging()


threshold = 0.3
fp_slack = 0.5
fn_slack = 0.05
relative_error = 0.1

alphabet = list(observations)

filtering_sul = FilteringSUL(
    mc, 
    initial_observation, 
    alphabet, 
    spec, 
    threshold, 
    horizon
)
eq_oracle = VerimonEqOracle(
    alphabet,
    filtering_sul,
    mc,
    threshold,
    fp_slack,
    fn_slack,
    horizon,
    spec,
    good_label,
    relative_error,
    False,
    expr_manager
)
learned_monitor: Dfa = run_Lstar(
    alphabet,
    filtering_sul,
    eq_oracle,
    automaton_type="dfa",
    print_level=2,
) #type: ignore

DEBUG:2024-11-26 16:56:56,777 - (0.00s) - MonitorLearning.py - Filtering SUL is using the following risk function: bit vector(290/699) [192 193 194 195 199 201 203 205 207 208 209 210 231 232 233 234 278 279 280 281 282 284 285 286 287 288 289 290 291 292 293 294 295 296 297 300 301 302 303 304 305 306 307 308 309 310 311 312 314 315 316 317 324 325 326 327 334 335 336 337 339 340 341 342 343 344 345 346 347 366 367 368 369 370 371 372 373 374 375 377 379 381 383 385 391 392 393 394 395 396 397 398 399 400 406 407 408 409 410 411 412 413 414 415 416 417 418 419 420 421 422 423 424 425 457 458 459 460 461 462 463 464 465 466 472 473 474 475 476 477 478 479 480 481 483 485 487 489 491 497 498 499 500 501 502 503 504 505 506 512 513 514 515 516 517 518 519 520 521 522 523 524 525 526 527 528 529 530 531 537 538 539 540 541 542 543 544 545 546 549 551 553 555 557 558 559 560 561 562 563 564 565 566 567 568 569 570 571 572 573 574 575 576 577 578 579 580 581 582 583 584 585 586 587 588 589 

In [ ]:
from verimon.loaders import aalpy_dfa_to_stormvogel
from verimon.verify import false_positive

mon_cycl = aalpy_dfa_to_stormvogel(learned_monitor)
# export_to_drn(stormvogel_to_stormpy(mon_cycl), "monitor.drn")
result_goal, trace, assignment, product = false_positive(mc, mon_cycl, horizon, expr_manager, options = {"good_spec": spec, "good_label": good_label})

DEBUG:2024-11-18 11:31:11,366 - (208.04s) - verify.py - Building model 
DEBUG:2024-11-18 11:31:11,369 - (0.00s) - verify.py - Unrolling done 
INFO:2024-11-18 11:31:11,383 - (0.01s) - generator.py - New good states become: [15, 19, 21, 36, 37, 38] 
DEBUG:2024-11-18 11:31:11,383 - (0.00s) - verify.py - Apply spec done 


DEBUG:2024-11-18 11:31:11,643 - (0.26s) - verify.py - creating product done 
DEBUG:2024-11-18 11:31:11,643 - (0.00s) - verify.py - Creating trace 
2024-11-18 11:31:11,846 - pomdp.py - constructed POMDP having 15 observations.
2024-11-18 11:31:11,852 - pomdp.py - unfolding 1-FSC template into POMDP...
2024-11-18 11:31:11,855 - pomdp.py - constructed quotient MDP having 374 states and 6757 actions.
2024-11-18 11:31:11,899 - statistic.py - synthesis initiated, design space: 1e16
2024-11-18 11:31:12,011 - synthesizer_ar.py - value 0.5003 achieved after 0.37 seconds
2024-11-18 11:31:12,097 - synthesizer_ar.py - value 0.5186 achieved after 0.45 seconds
2024-11-18 11:31:12,154 - synthesizer_ar.py - value 0.549 achieved after 0.51 seconds
2024-11-18 11:31:14,742 - synthesizer_ar.py - value 0.5571 achieved after 3.1 seconds
> progress 1.169%, elapsed 3 s, estimated 256 s, iters = {MDP: 605}, opt = 0.5571
2024-11-18 11:31:15,196 - synthesizer_ar.py - value 0.5882 achieved after 3.55 seconds
2024